In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.shape

(10000, 14)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [5]:
df["Geography"].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [7]:
df["Gender"].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [9]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [10]:
df.drop(columns=["RowNumber", "Surname", "CustomerId"], inplace=True)

In [11]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [12]:
df.shape

(10000, 11)

In [13]:
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)

In [14]:
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


## Split Features/target, then train/test split

In [15]:
X = df.drop(columns=["Exited"])
y = df["Exited"].values

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# This standardizes every feature to mean = 0, and std = 1. Feature like "balance"(range from 0 to 250,000) and HasCard(range from 0 to 1) have wildly different scales. Without scaling, the gradient updates for weights attached to large-scale features would dominate the loss landscape, making training unstable or painfully slow.

In [19]:
X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

## Build the Multi layer perceptron

In [22]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [23]:
model = Sequential()

# Hidden layer 1
model.add(Dense(11, activation="sigmoid", input_dim=11))
# Hidden layer 2
model.add(Dense(11, activation="sigmoid"))
# Output layer
model.add(Dense(1, activation="sigmoid"))

# Hidden layer 1, Neurons=11, Input=11 features, Activation=sigmoid
# Hidden layer 2, Neurons=11, Input=11(from hidden 1), Activation=sigmoid
# Output Layer, Neurons=1, Input=11(from hidden 2), Activation=sigmoid

D:\Complete ML\FastAPI\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 276 (1.08 KB)

 Trainable params: 276 (1.08 KB)

 Non-trainable params: 0 (0.00 B)

## Compile - Defining the loss and optimizer

In [25]:
model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])

# loss is BCE, used because this is binary classification
# Optimizer = "adam" --> A smarter version of gradient descent.
# accuracy --> just for human-readable monitoring, not used in the actual weight updates

## Train - Forward prop + backprop + gradient descent, looped

In [26]:
history = model.fit(X_train, y_train, batch_size=50, epochs=100, verbose=1, validation_split=0.2)

# This single line is the entire training loop you've been deriving by hand, automated:

Epoch 1/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2066 - loss: 1.0782 - val_accuracy: 0.2013 - val_loss: 0.7749
Epoch 2/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6439 - loss: 0.6486 - val_accuracy: 0.7987 - val_loss: 0.5663
Epoch 3/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5386 - val_accuracy: 0.7987 - val_loss: 0.5144
Epoch 4/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5128 - val_accuracy: 0.7987 - val_loss: 0.5021
Epoch 5/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5068 - val_accuracy: 0.7987 - val_loss: 0.4989
Epoch 6/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5058 - val_accuracy: 0.7987 - val_loss: 0.4982
Epoch 7/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5054 - val_accuracy: 0.7987 - val_loss: 0.4978
Epoch 8/100
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7934 - loss: 0.5052 - val_accu

In [27]:
model.layers[0].get_weights()

[array([[-0.2601787 , -0.31957692,  0.4364404 ,  0.23422603,  0.00084092,
         -0.08477104,  0.3732881 , -0.37651837, -0.44407076, -0.31806093,
          0.1285244 ],
        [ 0.381344  , -0.1890641 ,  0.5930985 , -0.4386287 ,  0.36552212,
         -0.34451675, -0.5036457 ,  0.23868369,  0.5150391 , -0.08821724,
          0.25847203],
        [-0.13570382,  0.1457816 ,  0.2775268 ,  0.46336028,  0.28454313,
         -0.49570063, -0.06192744,  0.35725614, -0.42361882, -0.29071018,
         -0.49579528],
        [-0.37977013, -0.21330029,  0.0074457 ,  0.23026848, -0.36165273,
         -0.261434  ,  0.28291792,  0.44485044,  0.15374903,  0.5132094 ,
         -0.05154423],
        [ 0.24573757,  0.25578973,  0.5337685 ,  0.21345541, -0.4592104 ,
          0.42799303, -0.08576089, -0.17968084,  0.48706725,  0.06359377,
          0.2826049 ],
        [ 0.13085288, -0.44121945,  0.5975632 , -0.294116  , -0.02814925,
          0.5014573 , -0.28035426,  0.28747383,  0.52047694, -0.2124970

In [28]:
model.layers[1].get_weights()

[array([[-2.82369345e-01,  3.76728438e-02, -2.39534542e-01,
          1.83606312e-01, -1.72302574e-02, -7.33739436e-01,
         -3.86857808e-01,  5.31021953e-01, -1.97490126e-01,
         -1.79377481e-01,  5.41225970e-01],
        [ 1.20983243e-01, -3.35750312e-01, -2.23344415e-01,
          1.29568264e-01,  1.39981285e-01,  8.04289654e-02,
          3.05331796e-01,  4.35649663e-01, -4.83080178e-01,
          2.49525666e-01, -4.28132750e-02],
        [-5.28728187e-01,  6.74832523e-01, -3.24587375e-01,
          9.09416229e-02, -5.63261449e-01,  1.12181515e-01,
          2.21807972e-01, -2.02054307e-01, -5.18211186e-01,
          1.15934186e-01, -5.36780357e-01],
        [ 8.37742910e-02, -2.47292861e-01, -3.14284831e-01,
         -4.24376540e-02, -3.77198756e-01, -4.68546897e-01,
          3.28044146e-01,  1.61793664e-01, -3.42267871e-01,
          3.11786115e-01,  3.43086004e-01],
        [-9.39508617e-01,  6.16998494e-01,  5.21116257e-02,
         -1.19726050e+00, -4.07873005e-01, -

In [29]:
model.predict(X_test_trf)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 897us/step


array([[0.20977397],
       [0.21257545],
       [0.22862145],
       ...,
       [0.1978731 ],
       [0.21101052],
       [0.19484703]], shape=(2000, 1), dtype=float32)

In [30]:
y_log = model.predict(X_test_trf)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step


In [31]:
y_pred = np.where(y_log >0.5, 1, 0)

In [32]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8035